# Advanced: Multi-Strategy Group Variability Analysis with PAMOLA.CORE

**Goal**: Master all 3 group analysis strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: MD5-based exact matching for structured data
- **Strategy 2**: MinHash-based fuzzy matching for text-heavy fields
- **Strategy 3**: Hybrid weighted analysis with custom thresholds
- Calculate advanced variance metrics across strategies
- Identify optimal aggregation candidates
- Production deployment patterns for large datasets

**Prerequisites:**
- Completed the simple notebook
- Understanding of group variance concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/analyzers/
        └── 02_group_analyzer_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.group import GroupAnalyzerOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Load larger dataset from examples/data_examples/group_analysis_data.csv
- Or generate realistic 1000-record sample with grouped structure
- Inspect data distributions and group sizes

**What you'll see:**
- File location and load status
- Dataset summary (1000 records, 7 columns)
- First 10 sample records showing group structure
- Column statistics (unique counts, ranges)
- Group size distribution (1-5 resumes per person)

**Dataset features:**
- 1000 resume records
- ~250 unique persons (multiple resumes per person)
- Varying field consistency within groups
- Mix of structured and text fields

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'group_analysis_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate person IDs (250 unique persons, 1-5 resumes each)
    person_ids = []
    for person_id in range(1, 251):
        n_resumes = np.random.choice([1, 2, 3, 4, 5], p=[0.2, 0.3, 0.25, 0.15, 0.1])
        person_ids.extend([person_id] * n_resumes)
    
    # Trim or pad to exactly 1000 records
    if len(person_ids) > n_records:
        person_ids = person_ids[:n_records]
    else:
        person_ids.extend([250] * (n_records - len(person_ids)))
    
    # Generate names with variations
    first_names = ['Alice', 'Bob', 'Charlie', 'Diana', 'Eve', 'Frank', 'Grace', 
                   'Henry', 'Iris', 'Jack', 'Karen', 'Leo', 'Mia', 'Nathan', 'Olivia']
    last_names = ['Smith', 'Jones', 'Brown', 'Wilson', 'Davis', 'Miller', 'Taylor', 
                  'Anderson', 'Thomas', 'Jackson', 'White', 'Harris', 'Martin', 'Garcia']
    
    # Job titles with hierarchy
    job_titles = [
        'Software Engineer', 'Senior Software Engineer', 'Lead Software Engineer',
        'Data Scientist', 'Senior Data Scientist', 'Lead Data Scientist',
        'Product Manager', 'Senior Product Manager', 'Director of Product',
        'UX Designer', 'Senior UX Designer', 'UX Lead',
        'DevOps Engineer', 'Senior DevOps Engineer', 'DevOps Architect'
    ]
    
    # Cities
    cities = ['San Francisco', 'New York', 'Seattle', 'Boston', 'Austin', 
              'Chicago', 'Denver', 'Portland', 'Los Angeles', 'Atlanta']
    
    # Skills (for text analysis)
    skill_sets = [
        'Python, Java, SQL, Docker, Kubernetes',
        'JavaScript, React, Node.js, MongoDB, AWS',
        'Python, TensorFlow, PyTorch, Pandas, Scikit-learn',
        'Product Management, Agile, JIRA, User Research, Analytics',
        'UI/UX Design, Figma, Sketch, Adobe XD, Prototyping',
        'DevOps, CI/CD, Jenkins, Terraform, Ansible'
    ]
    
    # Generate data with group consistency
    records = []
    person_data = {}  # Store base data per person
    
    for resume_id, person_id in enumerate(person_ids, 1):
        # First resume for this person - create base data
        if person_id not in person_data:
            base_first = np.random.choice(first_names)
            base_last = np.random.choice(last_names)
            base_title = np.random.choice(job_titles)
            base_city = np.random.choice(cities)
            base_exp = np.random.randint(1, 15)
            base_skills = np.random.choice(skill_sets)
            
            person_data[person_id] = {
                'first_name': base_first,
                'last_name': base_last,
                'title': base_title,
                'city': base_city,
                'experience': base_exp,
                'skills': base_skills
            }
        
        # Get base data
        base = person_data[person_id]
        
        # Add some variation to create realistic variance
        # 80% chance to keep original, 20% chance to vary
        first_name = base['first_name'] if np.random.random() > 0.1 else np.random.choice(first_names)
        last_name = base['last_name'] if np.random.random() > 0.05 else base['last_name'] + '-Smith'
        
        # Title may change (career progression)
        if np.random.random() > 0.3:
            title = base['title']
        else:
            # Career progression
            if 'Senior' not in base['title'] and 'Lead' not in base['title']:
                title = 'Senior ' + base['title']
            else:
                title = base['title']
        
        # Location may change
        city = base['city'] if np.random.random() > 0.2 else np.random.choice(cities)
        
        # Experience increases over time
        experience = base['experience'] + np.random.randint(0, 3)
        
        # Skills may expand
        if np.random.random() > 0.4:
            skills = base['skills']
        else:
            skills = base['skills'] + ', ' + np.random.choice(['Git', 'Linux', 'Docker', 'AWS', 'Azure'])
        
        records.append({
            'resume_id': resume_id,
            'person_id': person_id,
            'first_name': first_name,
            'last_name': last_name,
            'title': title,
            'city': city,
            'experience_years': experience,
            'skills': skills
        })
    
    df = pd.DataFrame(records)
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records (showing group structure):")
print(df.head(10))

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n👥 Group Structure:")
group_sizes = df.groupby('person_id').size()
print(f"  Unique persons: {df['person_id'].nunique()}")
print(f"  Avg resumes per person: {group_sizes.mean():.2f}")
print(f"  Max resumes per person: {group_sizes.max()}")
print(f"  Group size distribution: {dict(group_sizes.value_counts().sort_index())}")

print(f"\n💡 Perfect dataset for testing all 3 analysis strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field assignments and weights
   - `grouping_field`: Field to group by (default: "person_id")
   - `analysis_fields`: Dictionary of fields to analyze with weights
     - Higher weight = more important for variance calculation
2. Run to validate fields and setup environment

**What you'll see:**
- Grouping field validation (✓ exists, unique groups, records per group)
- Analysis fields validation (✓ all fields exist with unique counts)
- Task directory created (✓)
- TaskReporter initialized (✓)
- Config kwargs ready (✓)
- DataSource created (✓)

**This setup will be reused for all 3 strategies**

**Field configuration pattern:**
```python
FIELD_CONFIG = {
    "grouping_field": "person_id",      # Group by this field
    "analysis_fields": {                # Fields to analyze
        "first_name": 2,                # weight=2 (high importance)
        "last_name": 2,                 # weight=2 (high importance)
        "title": 1,                     # weight=1 (medium importance)
        "city": 1                       # weight=1 (medium importance)
    }
}
```

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "grouping_field": "person_id",      # Field to group records by
    "analysis_fields": {                # Fields to analyze with weights
        "first_name": 2,                # High importance
        "last_name": 2,                 # High importance  
        "title": 1,                     # Medium importance
        "city": 1                       # Medium importance
    }
}

# Validate grouping field exists in dataset
print("📋 Validating field configuration...\n")
grouping_field = FIELD_CONFIG["grouping_field"]
if grouping_field not in df.columns:
    raise ValueError(
        f"❌ Grouping field '{grouping_field}' not found!\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update FIELD_CONFIG['grouping_field']"
    )

print(f"✓ Grouping field: '{grouping_field}'")
print(f"  Unique groups: {df[grouping_field].nunique()}")
print(f"  Records per group: {len(df) / df[grouping_field].nunique():.2f} avg")

# Validate analysis fields exist
analysis_fields = FIELD_CONFIG["analysis_fields"]
missing_fields = [f for f in analysis_fields.keys() if f not in df.columns]
if missing_fields:
    raise ValueError(
        f"❌ Analysis fields not found: {missing_fields}\n"
        f"Available columns: {', '.join(df.columns)}\n"
        f"Please update FIELD_CONFIG['analysis_fields']"
    )

print(f"\n✓ Analysis fields (with weights):")
for field, weight in analysis_fields.items():
    print(f"  {field:20s}: weight={weight}, unique={df[field].nunique()}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy_group_analysis",
    description="Multi-strategy group variability analysis",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: MD5-Based Exact Matching

**How to use:**
- Uses MD5 hashing for text field comparison
- Best for structured data with exact value matching
- Run to execute MD5-based strategy

**Key parameters:**
- `hash_algorithm='md5'`: Exact matching via hashing
- `text_length_threshold=100`: Apply hashing to text >100 chars
- `variance_threshold=0.2`: Groups ≤0.2 variance are aggregatable
- `large_group_threshold=100`: Size for "large" group classification

**What you'll see:**
- Configuration confirmation
- Progress bar: validate → group → analyze → metrics → visualize → save
- Completion message with execution time (2-5 seconds)
- Aggregation statistics (groups eligible for merging)

**Note:** MD5 requires exact matches - typos/variations count as different values

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: MD5-BASED EXACT MATCHING")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: MD5-based",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = GroupAnalyzerOperation(
    # Core parameters
    field_name=FIELD_CONFIG["grouping_field"],
    fields_config=FIELD_CONFIG["analysis_fields"],
    
    # Strategy: MD5-based exact matching
    hash_algorithm='md5',
    text_length_threshold=100,
    
    # Variance thresholds
    variance_threshold=0.2,
    large_group_threshold=100,
    large_group_variance_threshold=0.05,
    
    # Processing settings
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Hash algorithm:           {operation_s1.hash_algorithm}")
print(f"  Text threshold:           {operation_s1.text_length_threshold} chars")
print(f"  Variance threshold:       {operation_s1.variance_threshold}")
print(f"  Large group threshold:    {operation_s1.large_group_threshold}")

print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_md5',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Extract key metrics
if result_s1.metrics:
    print(f"\n📊 Quick Stats:")
    print(f"   Total groups:         {result_s1.metrics.get('total_groups', 0):,}")
    print(f"   Groups to aggregate:  {result_s1.metrics.get('groups_to_aggregate', 0):,}")
    print(f"   Avg variance:         {result_s1.metrics.get('avg_variance', 0):.4f}")

## STRATEGY 2: MinHash-Based Fuzzy Matching

**How to use:**
- Uses MinHash for fuzzy text similarity detection
- Best for text-heavy data with typos/variations
- Run to execute MinHash-based strategy

**Key parameters:**
- `hash_algorithm='minhash'`: Fuzzy matching via MinHash
- `minhash_similarity_threshold=0.7`: 70% similarity threshold
- `text_length_threshold=50`: Lower threshold for text processing
- `variance_threshold=0.25`: Slightly higher for fuzzy matching

**What you'll see:**
- Configuration confirmation with MinHash settings
- Progress bar: validate → group → minhash analysis → metrics → visualize → save
- Completion message with execution time (5-10 seconds, slower than MD5)
- Similarity-based grouping statistics

**Note:** MinHash detects similar but not identical text - handles typos/variations better

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: MINHASH-BASED FUZZY MATCHING")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6, 
    description="Strategy 2: MinHash", 
    unit="steps"
)

operation_s2 = GroupAnalyzerOperation(
    # Core parameters
    field_name=FIELD_CONFIG["grouping_field"],
    fields_config=FIELD_CONFIG["analysis_fields"],
    
    # Strategy: MinHash-based fuzzy matching
    hash_algorithm='minhash',
    minhash_similarity_threshold=0.7,
    text_length_threshold=50,  # Lower threshold for text processing
    
    # Variance thresholds (slightly higher for fuzzy)
    variance_threshold=0.25,
    large_group_threshold=100,
    large_group_variance_threshold=0.08,
    
    # Processing settings
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Hash algorithm:           {operation_s2.hash_algorithm}")
print(f"  MinHash similarity:       {operation_s2.minhash_similarity_threshold}")
print(f"  Text threshold:           {operation_s2.text_length_threshold} chars")
print(f"  Variance threshold:       {operation_s2.variance_threshold}")

start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_minhash',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Extract key metrics
if result_s2.metrics:
    print(f"\n📊 Quick Stats:")
    print(f"   Total groups:         {result_s2.metrics.get('total_groups', 0):,}")
    print(f"   Groups to aggregate:  {result_s2.metrics.get('groups_to_aggregate', 0):,}")
    print(f"   Avg variance:         {result_s2.metrics.get('avg_variance', 0):.4f}")

## STRATEGY 3: Hybrid Weighted Analysis

**How to use:**
- Combines MD5 exact matching with custom field weights
- Stricter thresholds for large groups
- Run to execute hybrid strategy

**Key parameters:**
- `hash_algorithm='md5'`: Exact matching baseline
- `variance_threshold=0.15`: Stricter than Strategy 1
- `large_group_threshold=50`: Lower threshold for "large" groups
- `large_group_variance_threshold=0.03`: Very strict for large groups

**What you'll see:**
- Configuration confirmation with hybrid settings
- Progress bar: validate → group → weighted analysis → metrics → visualize → save
- Completion message with execution time (3-6 seconds)
- Differentiated statistics for normal vs large groups

**Note:** Optimal for production with mixed group sizes - applies context-aware thresholds

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: HYBRID WEIGHTED ANALYSIS")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6, 
    description="Strategy 3: Hybrid", 
    unit="steps"
)

operation_s3 = GroupAnalyzerOperation(
    # Core parameters
    field_name=FIELD_CONFIG["grouping_field"],
    fields_config=FIELD_CONFIG["analysis_fields"],
    
    # Strategy: Hybrid with stricter thresholds
    hash_algorithm='md5',
    text_length_threshold=100,
    
    # Variance thresholds (differentiated by size)
    variance_threshold=0.15,            # Stricter for normal groups
    large_group_threshold=50,           # Lower threshold for "large"
    large_group_variance_threshold=0.03, # Very strict for large groups
    
    # Processing settings
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Hash algorithm:           {operation_s3.hash_algorithm}")
print(f"  Normal variance threshold: {operation_s3.variance_threshold}")
print(f"  Large group threshold:    {operation_s3.large_group_threshold} records")
print(f"  Large group variance:     {operation_s3.large_group_variance_threshold}")

start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_hybrid',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Extract key metrics
if result_s3.metrics:
    print(f"\n📊 Quick Stats:")
    print(f"   Total groups:         {result_s3.metrics.get('total_groups', 0):,}")
    print(f"   Groups to aggregate:  {result_s3.metrics.get('groups_to_aggregate', 0):,}")
    print(f"   Avg variance:         {result_s3.metrics.get('avg_variance', 0):.4f}")

## Step 4: Strategy Comparison

**How to use:**
- Review execution times for all 3 strategies
- Compare aggregation potential and variance metrics
- Identify best strategy for your use case

**What you'll see:**
- Execution time per strategy (seconds)
- Aggregation comparison (groups eligible for merging)
- Variance comparison (avg across all groups)
- Total processing time

**Strategy selection guide:**
- **MD5**: Structured data + exact matching required
- **MinHash**: Text-heavy + typo tolerance needed
- **Hybrid**: Mixed data + context-aware thresholds

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (MD5):      {elapsed_s1:6.2f}s")
print(f"  Strategy 2 (MinHash):  {elapsed_s2:6.2f}s")
print(f"  Strategy 3 (Hybrid):   {elapsed_s3:6.2f}s")
print(f"  Total:                 {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

# Compare aggregation potential
if result_s1.metrics and result_s2.metrics and result_s3.metrics:
    print(f"\n📈 Aggregation Potential:")
    agg_s1 = result_s1.metrics.get('groups_to_aggregate', 0)
    agg_s2 = result_s2.metrics.get('groups_to_aggregate', 0)
    agg_s3 = result_s3.metrics.get('groups_to_aggregate', 0)
    total_groups = result_s1.metrics.get('total_groups', 1)
    
    print(f"  Strategy 1: {agg_s1:4d} groups ({agg_s1/total_groups*100:5.1f}%)")
    print(f"  Strategy 2: {agg_s2:4d} groups ({agg_s2/total_groups*100:5.1f}%)")
    print(f"  Strategy 3: {agg_s3:4d} groups ({agg_s3/total_groups*100:5.1f}%)")
    
    print(f"\n📊 Average Variance:")
    var_s1 = result_s1.metrics.get('avg_variance', 0)
    var_s2 = result_s2.metrics.get('avg_variance', 0)
    var_s3 = result_s3.metrics.get('avg_variance', 0)
    
    print(f"  Strategy 1: {var_s1:.4f}")
    print(f"  Strategy 2: {var_s2:.4f}")
    print(f"  Strategy 3: {var_s3:.4f}")

## Step 5: Detailed Artifact Review

**How to use:**
- Review all generated artifacts from each strategy
- Auto-loads NEWEST files from each category
- Displays metrics and visualizations inline

**What you'll see (per strategy):**
1. **Metrics JSON**: Total groups, aggregation stats, variance distribution, field metrics
2. **Visualizations**: Variance distribution histogram, field variability heatmap (first 2 displayed)

**Note:** Only newest files shown. data_types metrics excluded automatically

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_md5', 'Strategy 1: MD5-Based Exact Matching'),
    ('strategy2_minhash', 'Strategy 2: MinHash-Based Fuzzy Matching'),
    ('strategy3_hybrid', 'Strategy 3: Hybrid Weighted Analysis')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics Summary (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"\n✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"\n⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                
                # Extract metrics (handle both direct metrics and nested structure)
                if 'metrics' in data:
                    metrics = data['metrics']
                else:
                    metrics = data
                
                # Main metrics
                print("   " + "-" * 60)
                print(f"   Field analyzed          : {metrics.get('field', 'N/A')}")
                print(f"   Total groups            : {metrics.get('total_groups', 0):,}")
                print(f"   Total records           : {metrics.get('total_records', 0):,}")
                print(f"   Groups to aggregate     : {metrics.get('groups_to_aggregate', 0):,}")
                
                # Variance statistics
                avg_var = metrics.get('avg_variance', 0)
                max_var = metrics.get('max_variance', 0)
                print(f"\n   📊 Variance Statistics:")
                print(f"      Average variance     : {avg_var:.4f}")
                print(f"      Maximum variance     : {max_var:.4f}")
                
                # Field-level metrics
                if 'field_metrics' in metrics:
                    print(f"\n   📋 Field-Level Analysis:")
                    field_metrics = metrics['field_metrics']
                    
                    for field_name, field_data in list(field_metrics.items())[:5]:  # Show first 5
                        print(f"      • {field_name:20s}:")
                        print(f"        - Avg variance      : {field_data.get('avg_variance', 0):.4f}")
                        print(f"        - Unique values     : {field_data.get('unique_values_total', 0):,}")
                        print(f"        - Avg dup ratio     : {field_data.get('avg_duplication_ratio', 0):.2f}")
                    
                    if len(field_metrics) > 5:
                        print(f"      ... and {len(field_metrics) - 5} more fields")
                
                # Threshold distribution
                if 'threshold_metrics' in metrics:
                    print(f"\n   📈 Variance Distribution:")
                    threshold_metrics = metrics['threshold_metrics']
                    total = sum(threshold_metrics.values())
                    
                    for threshold, count in threshold_metrics.items():
                        pct = (count / total * 100) if total > 0 else 0
                        threshold_label = threshold.replace('_', ' ').replace('to', '-')
                        print(f"      • {threshold_label:15s}: {count:4,} groups ({pct:5.1f}%)")
                
                # Algorithm configuration
                if 'algorithm_info' in metrics:
                    print(f"\n   ⚙️  Algorithm Configuration:")
                    algo_info = metrics['algorithm_info']
                    
                    config_keys = [
                        ('hash_algorithm_used', 'Hash algorithm'),
                        ('variance_threshold', 'Variance threshold'),
                        ('minhash_similarity_threshold', 'MinHash threshold'),
                        ('text_length_threshold', 'Text length threshold')
                    ]
                    
                    for key, label in config_keys:
                        if key in algo_info:
                            value = algo_info[key]
                            if isinstance(value, float):
                                print(f"      • {label:25s}: {value:.2f}")
                            else:
                                print(f"      • {label:25s}: {value}")
                
                # Fields analyzed
                if 'fields_analyzed' in metrics:
                    fields = metrics['fields_analyzed']
                    print(f"\n   🔍 Fields Analyzed ({len(fields)}):")
                    print(f"      {', '.join(fields)}")
                
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Detailed Group Analysis (from output/)
    output_dir = strategy_dir / 'output'
    if output_dir.exists():
        group_files = sorted(
            list(output_dir.glob('*_group_analysis_*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if group_files:
            print(f"\n📊 GROUP ANALYSIS: {group_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(group_files[0], 'r') as f:
                    group_data = json.load(f)
                
                if 'summary' in group_data:
                    summary = group_data['summary']
                    print(f"   Total groups analyzed   : {summary.get('total_groups', 0):,}")
                    print(f"   High variance groups    : {summary.get('high_variance_groups', 0):,}")
                    print(f"   Recommended aggregations: {summary.get('recommended_aggregations', 0):,}")
                
                # Show sample groups
                if 'sample_groups' in group_data:
                    print(f"\n   Sample Groups (first 3):")
                    for idx, group in enumerate(group_data['sample_groups'][:3], 1):
                        print(f"      Group {idx}:")
                        print(f"        Records         : {group.get('total_records', 0)}")
                        print(f"        Variance        : {group.get('weighted_variance', 0):.4f}")
                        print(f"        Should aggregate: {group.get('should_aggregate', False)}")
                        
            except Exception as e:
                print(f"   ⚠️  Error reading group analysis: {e}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"\n   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 6: Calculate Privacy Metrics

**How to use:**
- Calculate group variance impact on privacy
- Compare aggregation safety across strategies
- Run AFTER all 3 strategies complete

**What you'll see:**
- Groups by variance range (per strategy)
- Safe aggregation count (variance ≤ threshold)
- Privacy risk assessment

**Privacy guidance:**
- Variance ≤ 0.1: Very low risk (highly similar records)
- Variance 0.1-0.2: Low risk (acceptable variation)
- Variance 0.2-0.5: Medium risk (careful review needed)
- Variance > 0.5: High risk (not recommended for aggregation)

In [ ]:
print("\n" + "=" * 80)
print("🔒 PRIVACY METRICS")
print("=" * 80 + "\n")

# Check if all strategies completed
try:
    if not (result_s1.metrics and result_s2.metrics and result_s3.metrics):
        print("⚠️  Run all 3 strategies first!")
    else:
        strategies = [
            ('Strategy 1 (MD5)', result_s1.metrics),
            ('Strategy 2 (MinHash)', result_s2.metrics),
            ('Strategy 3 (Hybrid)', result_s3.metrics)
        ]
        
        for name, metrics in strategies:
            print(f"📊 {name}:")
            
            # Get threshold distribution
            thresholds = metrics.get('threshold_metrics', {})
            
            if thresholds:
                print(f"\n   Variance Distribution:")
                labels = {
                    'below_0.1': '< 0.1 (Very Low Risk)',
                    '0.1_to_0.2': '0.1 - 0.2 (Low Risk)',
                    '0.2_to_0.5': '0.2 - 0.5 (Medium Risk)',
                    '0.5_to_0.8': '0.5 - 0.8 (High Risk)',
                    'above_0.8': '> 0.8 (Very High Risk)'
                }
                
                total = metrics.get('total_groups', 1)
                for key, label in labels.items():
                    count = thresholds.get(key, 0)
                    pct = (count / total) * 100
                    print(f"      {label:30s}: {count:4d} ({pct:5.1f}%)")
            
            # Aggregation safety
            agg_count = metrics.get('groups_to_aggregate', 0)
            total_groups = metrics.get('total_groups', 1)
            agg_pct = (agg_count / total_groups) * 100
            
            print(f"\n   Safe for Aggregation: {agg_count:,} of {total_groups:,} groups ({agg_pct:.1f}%)")
            print("\n" + "-" * 80 + "\n")
        
        print(f"\n💡 Lower variance = Higher privacy protection when aggregating")
        
except NameError:
    print("⚠️  Run Step 3 (Setup Environment) and all strategies first!")

## Step 7: Export Analysis Results

**How to use:**
- Export metrics summaries from all strategies
- Each strategy gets its own summary file
- Run AFTER all 3 strategies complete

**What you'll see (per strategy):**
- Summary statistics (total groups, aggregation potential, variance)
- Export confirmation with file path
- File size

**Output structure:**
```
advanced_output/
├── strategy1_md5/
│   ├── metrics/              # Full metrics JSON
│   ├── visualizations/       # Charts
│   └── summary.json          # Key stats for quick reference
├── strategy2_minhash/
│   └── ...
└── strategy3_hybrid/
    └── ...
```

In [ ]:
print("=" * 80)
print("📦 EXPORTING ANALYSIS RESULTS FROM ALL STRATEGIES")
print("=" * 80)

# Check if all strategies completed
try:
    if not (result_s1.metrics and result_s2.metrics and result_s3.metrics):
        print("\n❌ Run all 3 strategies first!")
    else:
        export_base_dir = task_dir
        print(f"\n📂 Export directory: {export_base_dir}\n")
        
        strategies = [
            ('strategy1_md5', 'MD5-Based', result_s1.metrics),
            ('strategy2_minhash', 'MinHash-Based', result_s2.metrics),
            ('strategy3_hybrid', 'Hybrid Weighted', result_s3.metrics)
        ]
        
        for dir_name, strategy_name, metrics in strategies:
            print("=" * 80)
            print(f"📊 STRATEGY: {strategy_name}")
            print("=" * 80)
            
            strategy_dir = export_base_dir / dir_name
            strategy_dir.mkdir(parents=True, exist_ok=True)
            
            # Create summary
            summary = {
                'strategy': strategy_name,
                'timestamp': datetime.now().isoformat(),
                'statistics': {
                    'total_groups': metrics.get('total_groups', 0),
                    'total_records': metrics.get('total_records', 0),
                    'groups_to_aggregate': metrics.get('groups_to_aggregate', 0),
                    'avg_variance': metrics.get('avg_variance', 0),
                    'max_variance': metrics.get('max_variance', 0)
                },
                'variance_distribution': metrics.get('threshold_metrics', {})
            }
            
            # Save summary
            summary_path = strategy_dir / 'summary.json'
            try:
                with open(summary_path, 'w') as f:
                    json.dump(summary, f, indent=2)
                print(f"\n✅ Saved: {summary_path}")
                print(f"   Size: {summary_path.stat().st_size / 1024:.1f} KB")
            except PermissionError:
                print(f"\n⚠️  Cannot save (file may be open): {summary_path}")
            
            # Display summary
            print(f"\n📊 Summary:")
            print(f"   Total groups:         {summary['statistics']['total_groups']:,}")
            print(f"   Total records:        {summary['statistics']['total_records']:,}")
            print(f"   Groups to aggregate:  {summary['statistics']['groups_to_aggregate']:,}")
            print(f"   Avg variance:         {summary['statistics']['avg_variance']:.4f}")
            print(f"   Max variance:         {summary['statistics']['max_variance']:.4f}")
            print()
        
        # Overall summary
        print("\n" + "=" * 80)
        print("✅ EXPORT COMPLETE")
        print("=" * 80)
        print(f"\n📂 All summaries saved to: {export_base_dir}")
        
        if 'elapsed_s1' in locals():
            total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
            print(f"⏱️  Total processing time: {total_time:.2f}s")
        
except NameError:
    print("❌ Run Step 3 (Setup Environment) and all strategies first!")

## 🎯 Summary

**Accomplished:**
- ✅ 3 strategies implemented and compared
- ✅ Group variance metrics calculated
- ✅ Aggregation potential assessed
- ✅ Production-ready artifacts generated

**Key takeaways:**
- MD5 is fastest but requires exact matches
- MinHash handles text variations but slower
- Hybrid balances performance and accuracy
- Lower variance = safer aggregation
- Large groups need stricter thresholds

**Strategy recommendations:**
- **Use MD5** when: Structured data, exact matching acceptable, speed critical
- **Use MinHash** when: Text-heavy data, typos/variations expected, accuracy > speed
- **Use Hybrid** when: Mixed data types, context-aware processing needed, production deployment

**Next steps:**
- Test with your own datasets
- Tune thresholds for optimal privacy/utility balance
- Integrate with anonymization pipeline
- Deploy to production environment

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)